In [2]:
import psycopg2 as psycopg
import pandas as pd

connection = {"sslmode": "require", "target_session_attrs": "read-write"}
postgres_credentials = {
    "host": "rc1b-uh7kdmcx67eomesf.mdb.yandexcloud.net",
    "port": "6432",
    "dbname": "playground_mle_20260614_fb07c65e05",
    "user": "mle_20260614_fb07c65e05",
    "password": "90eff80856bf46ee968ed1e0bdc85990",
}
assert all(
    [var_value != "" for var_value in list(postgres_credentials.values())]
)

connection.update(postgres_credentials)

# определим название таблицы, в которой хранятся наши данные.
TABLE_NAME = "users_churn"

# эта конструкция создаёт контекстное управление для соединения с базой данных
# оператор with гарантирует, что соединение будет корректно закрыто после выполнения всех операций
# закрыто оно будет даже в случае ошибки, чтобы не допустить "утечку памяти"
with psycopg.connect(**connection) as conn:

    # создаёт объект курсора для выполнения запросов к базе данных
    # с помощью метода execute() выполняется SQL-запрос для выборки данных из таблицы TABLE_NAME
    with conn.cursor() as cur:
        cur.execute(f"SELECT * FROM {TABLE_NAME}")

        # извлекаем все строки, полученные в результате выполнения запроса
        data = cur.fetchall()

        # получает список имён столбцов из объекта курсора
        columns = [col[0] for col in cur.description]

# создаёт объект DataFrame из полученных данных и имён столбцов.
# это позволяет удобно работать с данными в Python, используя библиотеку Pandas.
df = pd.DataFrame(data, columns=columns)

In [11]:
df.to_csv("users_churn.csv", index=False)

In [12]:
# 1. Название колонок вашего датафрейма запишите в текстовый файл
with open("columns.txt", "w", encoding="utf-8") as fio:
    fio.write("".join(df.columns))

In [10]:
col = "monthly_charges"
unique_values = df[col].unique().tolist()
unique_values

[29.85,
 56.95,
 53.85,
 42.3,
 70.7,
 99.65,
 89.1,
 29.75,
 104.8,
 56.15,
 49.95,
 18.95,
 100.35,
 103.7,
 105.5,
 113.25,
 20.65,
 106.7,
 55.2,
 90.05,
 39.65,
 19.8,
 20.15,
 59.9,
 59.6,
 55.3,
 99.35,
 30.2,
 90.25,
 64.7,
 96.35,
 95.5,
 66.15,
 20.2,
 45.25,
 99.9,
 69.7,
 74.8,
 106.35,
 97.85,
 49.55,
 69.2,
 20.75,
 79.85,
 76.2,
 84.5,
 49.25,
 80.65,
 79.75,
 64.15,
 99.1,
 69.5,
 74.85,
 95.45,
 108.45,
 24.95,
 107.5,
 100.5,
 89.9,
 42.1,
 54.4,
 94.4,
 75.3,
 78.9,
 79.2,
 49.05,
 20.4,
 111.6,
 24.25,
 64.5,
 110.5,
 55.65,
 54.65,
 74.75,
 25.9,
 79.35,
 50.55,
 75.15,
 103.8,
 99.3,
 62.15,
 19.95,
 33.75,
 82.05,
 74.7,
 84.0,
 111.05,
 100.9,
 78.95,
 66.85,
 21.05,
 21.0,
 98.5,
 19.45,
 95.0,
 45.55,
 110.0,
 24.3,
 104.15,
 30.15,
 94.35,
 19.4,
 96.75,
 57.95,
 91.65,
 76.5,
 54.6,
 89.85,
 31.05,
 100.25,
 85.2,
 99.8,
 20.7,
 74.4,
 50.7,
 20.85,
 88.95,
 78.05,
 23.55,
 19.75,
 56.45,
 85.95,
 58.6,
 35.45,
 44.35,
 25.7,
 75.0,
 19.6,
 70.45,
 88.05,
 7

In [3]:
counts_columns = [
    "type", "paperless_billing", "internet_service", "online_security", "online_backup", "device_protection",
    "tech_support", "streaming_tv", "streaming_movies", "gender", "senior_citizen", "partner", "dependents",
    "multiple_lines", "target"
]

stats = {}

for col in counts_columns:
    # считаем частоты значений и добавляем префикс с названием колонки в ключ
    column_stat = df[col].value_counts(dropna=False).to_dict()
    column_stat = {f"{col}_{key}": value for key, value in column_stat.items()}

    # обновляем словарь stats
    stats.update(column_stat)

monthly_charges = pd.to_numeric(df["monthly_charges"], errors="coerce")
total_charges = pd.to_numeric(df["total_charges"], errors="coerce")

stats["data_length"] = df.shape[0]
stats["monthly_charges_min"] = monthly_charges.min()
stats["monthly_charges_max"] = monthly_charges.max()
stats["monthly_charges_mean"] = monthly_charges.mean()
stats["monthly_charges_median"] = monthly_charges.median()
stats["total_charges_min"] = total_charges.min()
stats["total_charges_max"] = total_charges.max()
stats["total_charges_mean"] = total_charges.mean()
stats["total_charges_median"] = total_charges.median()
stats["unique_customers_number"] = df["customer_id"].nunique()
stats["end_date_nan"] = (df["end_date"].isna() | (df["end_date"] == "")).sum()

stats

{'type_Month-to-month': 3875,
 'type_Two year': 1695,
 'type_One year': 1473,
 'paperless_billing_Yes': 4171,
 'paperless_billing_No': 2872,
 'internet_service_Fiber optic': 3096,
 'internet_service_DSL': 2421,
 'internet_service_nan': 1526,
 'online_security_No': 3498,
 'online_security_Yes': 2019,
 'online_security_nan': 1526,
 'online_backup_No': 3088,
 'online_backup_Yes': 2429,
 'online_backup_nan': 1526,
 'device_protection_No': 3095,
 'device_protection_Yes': 2422,
 'device_protection_nan': 1526,
 'tech_support_No': 3473,
 'tech_support_Yes': 2044,
 'tech_support_nan': 1526,
 'streaming_tv_No': 2810,
 'streaming_tv_Yes': 2707,
 'streaming_tv_nan': 1526,
 'streaming_movies_No': 2785,
 'streaming_movies_Yes': 2732,
 'streaming_movies_nan': 1526,
 'gender_Male': 3555,
 'gender_Female': 3488,
 'senior_citizen_0': 5901,
 'senior_citizen_1': 1142,
 'partner_No': 3641,
 'partner_Yes': 3402,
 'dependents_No': 4933,
 'dependents_Yes': 2110,
 'multiple_lines_No': 3390,
 'multiple_lines_Ye

In [19]:
import os

import mlflow

TRACKING_SERVER_HOST = "127.0.0.1"
TRACKING_SERVER_PORT = 5000

YOUR_NAME = "den"  # введите своё имя для создания уникального эксперимента

# название тестового эксперимента и запуска (run) внутри него
EXPERIMENT_NAME = f"test_connection_experiment_{YOUR_NAME}1"
RUN_NAME = "test_connection_run_1"

# тестовые данные
METRIC_NAME = "test_metric"
METRIC_VALUE = 0

# устанавливаем host, который будет отслеживать наши эксперименты
mlflow.set_tracking_uri(
    f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}"
)

# создаём тестовый эксперимент и записываем в него тестовую информацию
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
if not experiment:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)
else:
    experiment_id = experiment.experiment_id
print(f"Experiment ID: {experiment_id}")

with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    run_id = run.info.run_id
    mlflow.log_metric(METRIC_NAME, METRIC_VALUE)

Experiment ID: 3
🏃 View run test_connection_run_1 at: http://127.0.0.1:5000/#/experiments/3/runs/118ba5ec0e4747328b56d944af175500
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


In [ ]:
# задаём название эксперимента и имя запуска для логирования в MLflow

EXPERIMENT_NAME = "churn_fio"
RUN_NAME = "data_check"

# создаём новый эксперимент в MLflow с указанным названием
# если эксперимент с таким именем уже существует,
# MLflow возвращает идентификатор существующего эксперимента
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
if not experiment:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)
else:
    experiment_id = experiment.experiment_id

# гарантируем наличие файлов перед логированием артефактов
if not os.path.exists('columns.txt'):
    with open('columns.txt', 'w', encoding='utf-8') as fio:
        fio.write(''.join(df.columns))
if not os.path.exists('users_churn.csv'):
    df.to_csv('users_churn.csv', index=False)

with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    # получаем уникальный идентификатор запуска эксперимента
    run_id = run.info.run_id
    
    # логируем метрики эксперимента
    # предполагается, что переменная stats содержит словарь с метриками,
    # объявлять переменную stats не надо,
    # где ключи — это названия метрик, а значения — числовые значения метрик
    mlflow.log_metrics(stats)
    
    # логируем файлы как артефакты эксперимента — 'columns.txt' и 'users_churn.csv'
    mlflow.log_artifact('columns.txt', artifact_path='dataframe')
    mlflow.log_artifact('users_churn.csv', artifact_path='dataframe')


experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
# получаем данные о запуске эксперимента по его уникальному идентификатору
run = mlflow.get_run(run_id)

# проверяем, что статус запуска эксперимента изменён на 'FINISHED'
# это утверждение (assert) можно использовать для автоматической проверки того,
# что эксперимент был завершён успешно
assert run.info.status == 'FINISHED'

# удаляем файлы 'columns.txt' и 'users_churn.csv' из файловой системы,
# чтобы очистить рабочую среду после логирования артефактов
os.remove('columns.txt')
os.remove('users_churn.csv')